# Standard name definitions

This notebook attempts to define standard names for quantities in a subset of
the IDSs from DD version `4.0.0`:
- pf_active
- pf_passive
- equilibrium
- wall
- magnetics
- tf
- core_profiles

Requirements:
- IMASPy
  
  **N.B.** this notebook is compatible with the open-sourced `imas-python` as
  well, but you'll need to update the imports in the first cell to reflect that:

  ```python
  # To use the open-source imas-python:
  import imas as imaspy
  from imas.ids_data_type import IDSDataType
  ```
- [pandas](https://pypi.org/project/pandas/)

This notebook was made for exploration and research and specific to the IDSs &
DD version listed above.

# DD import

In [ ]:
import imaspy
from imaspy.ids_data_type import IDSDataType
import pandas
import re

In [ ]:
ids_names = ["pf_active", "pf_passive", "equilibrium", "wall", "magnetics", "tf", "core_profiles"]
dd_version = "4.0.0"

factory = imaspy.IDSFactory(dd_version)

In [ ]:
def iter_dd_quantities(ids_name):
    ids = factory.new(ids_name)

    def inner(metadata):
        for child in metadata:
            if (
                # Leaf nodes
                child.data_type not in (IDSDataType.STRUCTURE, IDSDataType.STRUCT_ARRAY)
                # Or GGD value container structures
                or "generic_grid_" in getattr(child, "structure_reference", None)
            ):
                yield (
                    ids_name,
                    child.path_string,
                    child.units,
                    child.documentation,
                    child.data_type.value,
                    getattr(child, "structure_reference", None)
                )
            else:
                yield from inner(child)

    yield from inner(ids.metadata)

In [ ]:
df = pandas.DataFrame(
    (data for ids_name in ids_names for data in iter_dd_quantities(ids_name)),
    columns=("ids_name", "dd_path", "units", "documentation", "data_type", "structure_reference"),
)
# df = df.set_index(["ids_name", "dd_path"])

# Add columns for standard names and modifier(s)
df["standard_name"] = None
df["modifier"] = None
df["cell_methods"] = None
df["notes"] = None
df

# Define standard names, modifiers and cell methods

## Anything without standard names

In [ ]:
# ids_properties and code structures don't contain physical quantities
rows = df["dd_path"].str.startswith(("ids_properties/", "code/"))
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "ids_properties / code structures don't contain physical quantities"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Identifiers and names/labels
rows = todo & (df["dd_path"].str.endswith(("name", "/description")) | df["dd_path"].str.contains("/identifier"))
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "Name, description or identifier"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# material identifier
rows = todo & df["dd_path"].str.endswith("material/grid_subset")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "Materials are described with identifiers, no standard names"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Indices, ideally these should be handled as shared dimensions
rows = todo & (df["dd_path"].str.endswith(
    ("index", "indices", "indices_differential", "indices_compound", "ion_time_slice")))
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "Index, not a physical quantity"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Types
rows = todo & df["dd_path"].str.endswith(("type", "types"))
display(df[rows])
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "Type specifier, not a physical quantity"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# URIs / paths to other data elements
rows = todo & df["dd_path"].str.endswith(("uri", "path", "source"))
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "URI or path to other data element"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# XML parameters
rows = todo & df["documentation"].str.contains("XML")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "XML parameters, not a physical quantity"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Flags, these should probably be attributes of quantities in the fusion conventions?
rows = todo & df["documentation"].str.contains("flag|validity") | (df["dd_path"] == "coils_n")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "Flags describing other data"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Error bars
rows = todo & df["dd_path"].str.endswith(("_error_upper", "_error_lower"))
df.loc[rows, "standard_name"] = "<see related non-error quantity>"
df.loc[rows, "modifier"] = "standard_error"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Topologies
rows = todo & df["dd_path"].str.endswith(("circuit/connections", "contour_tree/edges"))
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "Topology / connectivity matrix, not a physical quantity"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Algorithm convergence is not a physical quantity
rows = todo & df["dd_path"].str.contains("convergence/iterations_n")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "Algorithm convergence is not a physical quantity (and solver-dependent)"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# 3dB bandwidth doesn't map to one standard name, it's a 2-element quantity
# (lower-frequency, upper-frequency)
rows = todo & df["dd_path"].str.endswith("bandwidth_3db")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "Maps to two physical properties of the diagnostic: upper and lower frequency bound of the band pass filter."

todo = df["standard_name"].isnull()
todo.sum()

## Ambigiuous or too generic substructures

In [ ]:
# Miscellaneous string descriptions
rows = todo & (df["dd_path"] == "supply/nonlinear_model")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "Description, not a physical quantity"

# molecule vibrations and electron configuration, these are STR_0D and/or not clearly
# defined by DD
rows = todo & df["dd_path"].str.endswith(
    ("state/vibrational_level", "state/vibrational_mode", "state/electron_configuration"))
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "This ion/neutral state description is not clearly defined by the DD, so we don't assign a standard name at this point."

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# GGD grids
rows = todo & df["dd_path"].str.endswith(("grid", "grid_ggd"))
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "GGD Grid"

todo = df["standard_name"].isnull()
todo.sum()
df[rows]

In [ ]:
# *_fit structures in core_profiles and constraints structure in equilibrium.
# To discuss: should these have standard names?
rows = todo & df["dd_path"].str.contains("_fit|/constraints/")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "To discuss how to handle standard names for fitting parameters..."

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Mixed units cannot be mapped
rows = todo & ((df["units"] == "mixed") | df["units"].str.startswith("as_parent for a local"))
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "No clear definition, interpretation depends on other DD quantities (e.g. identifiers)"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# core_profiles/statistics substructure is too generic to apply standard names
rows = todo & df["dd_path"].str.contains("statistics/")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "statistics substructures are too generic for standard names"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Miscellaneous
rows = todo & df["dd_path"].str.endswith("global_quantities/length_pol")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "Poloidal length of **the magnetic surface** -> which magnetic surface?"

rows = todo & df["dd_path"].str.contains("power_density_(?:inner|outer)_target_max")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "Named power density, but with units of power [W]... not clear what this should be."

rows = todo & (df["dd_path"] == "description_2d/vessel/unit/element/j_phi/data")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "Named `j` (current density), but description and units indicate current."


todo = df["standard_name"].isnull()
todo.sum()

## Time

In [ ]:
# All global and local time arrays
rows = todo & df["dd_path"].str.endswith("time")
df.loc[rows, "standard_name"] = "time"

todo = df["standard_name"].isnull()
todo.sum()

## Latency / delay

In [ ]:
rows = todo & df["dd_path"].str.contains("latency|supply/delay")
df.loc[rows, "standard_name"] = "latency"
df.loc[rows, "notes"] = (
    "Perhaps this should be a more specific standard name reflecting that this is the "
    "(maximum) delay between input command and actuator response?"
)

## Charges

In [ ]:
# Nuclear charge
rows = todo & df["dd_path"].str.endswith("z_n")
df.loc[rows, "standard_name"] = "nuclear_charge"
# Ion charge
rows = todo & df["dd_path"].str.endswith(("z_ion", "z_ion_1d"))
df.loc[rows, "standard_name"] = "particle_charge"
df.loc[rows, "cell_methods"] = "charge_state: mean"  # To discuss, it's actually a weighted average
# Squared ion charge
rows = todo & df["dd_path"].str.endswith("z_ion_square_1d")
df.loc[rows, "standard_name"] = "square_of_particle_charge"
df.loc[rows, "cell_methods"] = "charge_state: mean"  # To discuss, it's actually a weighted average
# Ion charge state bundles
rows = todo & df["dd_path"].str.endswith("z_min")
df.loc[rows, "standard_name"] = "particle_charge"
df.loc[rows, "cell_methods"] = "charge_state: minimum"  # To discuss
rows = todo & df["dd_path"].str.endswith("z_max")
df.loc[rows, "standard_name"] = "particle_charge"
df.loc[rows, "cell_methods"] = "charge_state: maximum"  # To discuss
rows = todo & df["dd_path"].str.endswith(("z_average", "z_average_1d"))
df.loc[rows, "standard_name"] = "particle_charge"
df.loc[rows, "cell_methods"] = "charge_state: mean"  # To discuss, it's actually a weighted average
rows = todo & df["dd_path"].str.endswith(("z_square_average", "z_average_square_1d"))
df.loc[rows, "standard_name"] = "square_of_particle_charge"
df.loc[rows, "cell_methods"] = "charge_state: mean"  # To discuss, it's actually a weighted average

# Effective charge
rows = todo & df["dd_path"].str.endswith("zeff")
df.loc[rows, "standard_name"] = "effective_charge"

todo = df["standard_name"].isnull()
todo.sum()

## Resistances

In [ ]:
# Plasma resistance
rows = todo & df["dd_path"].str.endswith("plasma_resistance")
df.loc[rows, "standard_name"] = "plasma_resistance"

# All other resistances in the investigated IDSs have clear dimensions (e.g. coils,
# loop, supply, etc.) to indicate where the resistance applies to:
rows = todo & df["dd_path"].str.contains("resistance")
df.loc[rows, "standard_name"] = "resistance"

# Resistivities of loop and wall elements
rows = todo & df["dd_path"].str.endswith("resistivity")
df.loc[rows, "standard_name"] = "resistivity"

todo = df["standard_name"].isnull()
todo.sum()

## Magnetic fluxes

### Toroidal / poloidal (normalized) flux

In [ ]:
# Poloidal fluxes

# All /psi are poloidal flux in these IDSs
rows = todo & df["dd_path"].str.endswith("/psi")
df.loc[rows, "standard_name"] = "poloidal_flux"
# Some need a modifier as well, we'll adress those below:
rows = todo & df["dd_path"].str.endswith("q_min/psi")
df.loc[rows, "modifier"] = "at_minimum_q"

rows = todo & df["dd_path"].str.endswith(("boundary/psi", "psi_boundary"))
df.loc[rows, "standard_name"] = "poloidal_flux"
df.loc[rows, "modifier"] = "at_boundary"

rows = todo & df["dd_path"].str.endswith(("psi_axis", "psi_magnetic_axis"))
df.loc[rows, "standard_name"] = "poloidal_flux"
df.loc[rows, "modifier"] = "at_magnetic_axis"


# Normalized poloidal flux
rows = todo & df["dd_path"].str.endswith("psi_norm")
df.loc[rows, "standard_name"] = "normalized_poloidal_flux"
# Add modifier:
rows = todo & df["dd_path"].str.endswith("q_min/psi_norm")
df.loc[rows, "modifier"] = "at_minimum_q"


# Toroidal flux query is more specific to avoid matching all angles called phi
rows = todo & df["dd_path"].str.endswith(("ggd/phi", "profiles_1d/phi", "profiles_2d/phi"))
df.loc[rows, "standard_name"] = "toroidal_flux"


# N.B. there are no normalized toroidal fluxes in these IDSs

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# External poloidal flux
rows = todo & df["dd_path"].str.endswith("/psi_external_average")
df.loc[rows, "standard_name"] = "poloidal_flux"
df.loc[rows, "modifier"] = "from_external_circuits"
df.loc[rows, "cell_methods"] = "area: mean"
# To discuss:
df.loc[rows, "notes"] = "Which area to average over is not very clear, probably everything inside the boundary?"

# And its time derivative
rows = todo & df["dd_path"].str.endswith("/v_external")
df.loc[rows, "standard_name"] = "negative_tendency_of_poloidal_flux"  # hmmm
df.loc[rows, "modifier"] = "from_external_circuits"
df.loc[rows, "cell_methods"] = "area: mean"
# To discuss:
df.loc[rows, "notes"] = "Which area to average over is not very clear, probably everything inside the boundary?"

todo = df["standard_name"].isnull()
todo.sum()

### Toroidal / poloidal flux coordinates

In [ ]:
# rho_tor
rows = todo & df["dd_path"].str.endswith("/rho_tor")
df.loc[rows, "standard_name"] = "toroidal_flux_coordinate"

rows = todo & df["dd_path"].str.endswith("/rho_tor_boundary")
df.loc[rows, "standard_name"] = "toroidal_flux_coordinate"
df.loc[rows, "modifier"] = "at_boundary"

# rho_tor_norm
rows = todo & df["dd_path"].str.endswith("/rho_tor_norm")
df.loc[rows, "standard_name"] = "normalized_toroidal_flux_coordinate"
# Add modifier:
rows = todo & df["dd_path"].str.endswith("q_min/rho_tor_norm")
df.loc[rows, "modifier"] = "at_minimum_q"

# N.B. there is no rho_pol
# rho_pol_norm
rows = todo & df["dd_path"].str.endswith("/rho_pol_norm")
df.loc[rows, "standard_name"] = "normalized_poloidal_flux_coordinate"

# df.loc[df["standard_name"].str.contains("flux_coordinate", na=False)]

todo = df["standard_name"].isnull()
todo.sum()

### Flux diagnostics

In [ ]:
rows = df["dd_path"] == "flux_loop/flux/data"
df.loc[rows, "standard_name"] = "magnetic_flux"
df.loc[rows, "notes"] = "Perhaps 'enclosed_magnetic_flux' to be more precise?"

rows = df["dd_path"] == "diamagnetic_flux/data"
df.loc[rows, "standard_name"] = "diamagnetic_flux"  # to discuss
df.loc[rows, "notes"] = "Perhaps 'enclosed_diamagnetic_flux' to be more precise?"

todo = df["standard_name"].isnull()
todo.sum()

## Geometric data

In [ ]:
# Surface area of flux surfaces
rows = todo & df["dd_path"].str.endswith("/surface")
df.loc[rows, "standard_name"] = "surface_area"

# First wall surface area
rows = todo & df["dd_path"].str.endswith("surface_area")
df.loc[rows, "standard_name"] = "surface_area"
df.loc[rows, "modifier"] = "of_first_wall"  # ?

# Cross sections
rows = todo & df["dd_path"].str.endswith("/area") & (df["ids_name"] != "magnetics")
df.loc[rows, "standard_name"] = "cross_sectional_area"  # ?

# Inconsistent /area for different sensors, but that's what it is:
rows = todo & df["dd_path"].str.endswith(("b_field_phi_probe/area", "b_field_pol_probe/area"))
df.loc[rows, "standard_name"] = "area_per_turn"
rows = todo & df["dd_path"].str.endswith(("flux_loop/area", "rogowski_coil/area"))
df.loc[rows, "standard_name"] = "effective_area"

# Coil windings
rows = todo & df["dd_path"].str.endswith(("/turns_with_sign", "/turns"))
df.loc[rows, "standard_name"] = "number_of_coil_turns"  # ?
# Why are all these sensor structures slightly different? :(
rows = todo & df["dd_path"].str.endswith("/turns_per_metre")
df.loc[rows, "standard_name"] = "ratio_of_number_of_coil_turns_to_length"  # ?

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Volumes
rows = todo & df["dd_path"].str.contains("/volume")
df.loc[rows, "standard_name"] = "volume"

# First wall surface area
rows = todo & df["dd_path"].str.endswith("enclosed_volume")
df.loc[rows, "standard_name"] = "volume"
df.loc[rows, "modifier"] = "enclosed_by_first_wall"  # ???

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Blanket /r, /z and /phi: can check later if we need to add modifiers or more specific
# standard names
rows = todo & df["dd_path"].str.endswith("/r")
df.loc[rows, "standard_name"] = "radial_distance"

rows = todo & df["dd_path"].str.endswith("/z")
df.loc[rows, "standard_name"] = "vertical_distance"

rows = todo & df["dd_path"].str.endswith("/phi")  # toroidal flux phi is already done :)
df.loc[rows, "standard_name"] = "azimuth"

rows = todo & df["dd_path"].str.endswith(("/theta", "/poloidal_angle"))
df.loc[rows, "standard_name"] = "poloidal_angle"

# Toroidal angle is a local angle, whereas azimuth is the angle in the (global)
# cylindrical coordinate system.
rows = todo & df["dd_path"].str.endswith("/toroidal_angle")
df.loc[rows, "standard_name"] = "toroidal_angle"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Ignore geometry descriptions (rectangle, oblique, annulus, arc, etc.) for now...
rows = todo & df["dd_path"].str.contains("/geometry/|/conductor/cross_section/")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "Ignoring geometry descriptions for now."

display(rows.sum())
display(df.loc[rows])

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Ignore phi_extensions for now, it doesn't represent a single physical quantity
rows = todo & df["dd_path"].str.endswith("/phi_extensions")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "First and second dimension don't have the same physical meaning"

display(df.loc[rows])

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Lengths and thicknesses
rows = todo & df["dd_path"].str.endswith("/length")
df.loc[rows, "standard_name"] = "length"


rows = todo & df["dd_path"].str.contains("/thickness")
df.loc[rows, "standard_name"] = "thickness"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Reference major radius (r0), we have two in vacuum_toroidal_field structures and one
# in tf
rows = todo & df["dd_path"].str.endswith("r0")
df.loc[rows, "standard_name"] = "reference_radial_distance"

# Process vacuum_toroidal_field/b0 as well
rows = todo & df["dd_path"].str.endswith("/b0")
df.loc[rows, "standard_name"] = "vacuum_magnetic_field"
df.loc[rows, "modifier"] = "at_reference_radial_distance  toroidal" # to discuss

todo = df["standard_name"].isnull()
todo.sum()

## Magnetic fields

In [ ]:
# Components
rows = todo & df["dd_path"].str.endswith(("b_field_r", "b_field_phi", "b_field_z"))
df.loc[rows, "standard_name"] = "magnetic_field"
rows = todo & df["dd_path"].str.endswith("b_field_r")
df.loc[rows, "modifier"] = "radial"
rows = todo & df["dd_path"].str.endswith("b_field_phi")
df.loc[rows, "modifier"] = "toroidal"  # or azimuthal?
rows = todo & df["dd_path"].str.endswith("b_field_z")
df.loc[rows, "modifier"] = "vertical"

# vacuum magnetic field in tf
rows = todo & df["dd_path"].str.contains("field_map/b_field")
df.loc[rows, "standard_name"] = "vacuum_magnetic_field"

# flux surface B fields in profiles_1d
rows = todo & df["dd_path"].str.contains("profiles_1d/b_field_average")
df.loc[rows, "standard_name"] = "magnitude_of_magnetic_field"
df.loc[rows, "cell_methods"] = "area: mean"
rows = todo & df["dd_path"].str.contains("profiles_1d/b_field_min")
df.loc[rows, "standard_name"] = "magnitude_of_magnetic_field"
df.loc[rows, "cell_methods"] = "area: minimum"
rows = todo & df["dd_path"].str.contains("profiles_1d/b_field_max")
df.loc[rows, "standard_name"] = "magnitude_of_magnetic_field"
df.loc[rows, "cell_methods"] = "area: maximum"

# Measured b-fields
rows = todo & df["dd_path"].str.endswith("field/data")
df.loc[rows, "standard_name"] = "magnetic_field"
# it should be clear from the dimension (b_field_pol_probe or b_field_phi_probe) which
# component of the b-field is measured and where the probe is

# Ignore non-linear response functions for now
rows = todo & df["dd_path"].str.contains("non_linear_response")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "No standard names yet for response functions"

display(df.loc[rows])

todo = df["standard_name"].isnull()
todo.sum()


## Magnetic vector potential

In [ ]:
rows = todo & df["dd_path"].str.contains("a_field")
df.loc[rows, "standard_name"] = "magnetic_vector_potential"

# r/z/tor components in tf field_map
df.loc[df["dd_path"].str.endswith("a_field_r"), "modifier"] = "radial"
df.loc[df["dd_path"].str.endswith("a_field_z"), "modifier"] = "vertical"
df.loc[df["dd_path"].str.endswith("a_field_tor"), "modifier"] = "toroidal"

todo = df["standard_name"].isnull()
todo.sum()

## Densities

In [ ]:
# All density, density_fast and density_thermal
rows = todo & df["dd_path"].str.contains("/density")
df.loc[rows, "standard_name"] = "density"

# Modifiers for density_thermal and density_fast
rows = todo & df["dd_path"].str.contains("/density_thermal")
df.loc[rows, "modifier"] = "thermal"
rows = todo & df["dd_path"].str.contains("/density_fast")
df.loc[rows, "modifier"] = "fast"  # or non_thermal?
df.loc[rows, "notes"] = "Modifier could also be non_thermal?"

# Ion species and states have a dimension describing what the density applies to
# electrons don't, so we add a modifier
rows = todo & df["dd_path"].str.contains("electrons/density")
df.loc[rows, "modifier"] = "of_electrons"  # To discuss: could also use electron_density as standard name

# Averaged densities
rows = todo & df["dd_path"].str.endswith("n_i_thermal_total")
df.loc[rows, "standard_name"] = "density"
df.loc[rows, "cell_methods"] = "ion: sum"

rows = todo & df["dd_path"].str.endswith("n_i_volume_average")
df.loc[rows, "standard_name"] = "density"
df.loc[rows, "cell_methods"] = "ion: sum volume: average"

rows = todo & df["dd_path"].str.endswith("n_e_volume_average")
df.loc[rows, "standard_name"] = "density"
df.loc[rows, "modifier"] = "of_electrons"
df.loc[rows, "cell_methods"] = "volume: average"

# ratio of n_e and n_i
rows = todo & df["dd_path"].str.endswith("n_i_total_over_n_e")
df.loc[rows, "standard_name"] = "ratio_of_ion_density_to_electron_density"


todo = df["standard_name"].isnull()
todo.sum()

## Temperatures

In [ ]:
# Temperature of reactor components
rows = todo & (df["units"] == "K")
df.loc[rows, "standard_name"] = "temperature"

# Temperature of plasma species
rows = todo & (df["ids_name"] == "core_profiles") & df["dd_path"].str.endswith("temperature")
df.loc[rows, "standard_name"] = "plasma_temperature"

# electron temperature has no dimension to indicate it applies to electrons
rows = todo & df["dd_path"].str.endswith("electrons/temperature")
df.loc[rows, "modifier"] = "of_electrons"  # To discuss: could also use electron_temperature as standard name

# averaged temperatures
rows = todo & df["dd_path"].str.endswith("t_i_average")
df.loc[rows, "standard_name"] = "plasma_temperature"
df.loc[rows, "cell_methods"] = "ion: average"

rows = todo & df["dd_path"].str.endswith("t_e_volume_average")
df.loc[rows, "standard_name"] = "plasma_temperature"
df.loc[rows, "modifier"] = "of_electrons"  # To discuss: could also use electron_temperature as standard name
df.loc[rows, "cell_methods"] = "volume: average"

rows = todo & df["dd_path"].str.endswith("t_i_volume_average")  # these have an ion dimension still
df.loc[rows, "standard_name"] = "plasma_temperature"
df.loc[rows, "cell_methods"] = "volume: average"


todo = df["standard_name"].isnull()
todo.sum()

## Energies

In [ ]:
# Ioniziaton potential
rows = todo & df["dd_path"].str.endswith("ionization_potential")
df.loc[rows, "standard_name"] = "ionization_potential"

# MHD energy
rows = todo & df["dd_path"].str.endswith(("energy_mhd", "energy_diamagnetic"))  # TO check, do these actually have the same definition?
df.loc[rows, "standard_name"] = "plasma_energy_content"

todo = df["standard_name"].isnull()
todo.sum()

## Velocity and e_field profiles

In [ ]:
# Velocity
rows = todo & df["dd_path"].str.contains("/velocity/")
df.loc[rows, "standard_name"] = "velocity"  # perhaps particle_velocity, but that should be clear from the ion dimension

# E-field
rows = todo & df["dd_path"].str.contains("/e_field")
df.loc[rows, "standard_name"] = "electric_field"

# Add modifiers for the components
rows = todo & df["dd_path"].str.endswith("/radial")
df.loc[rows, "modifier"] = "radial"
rows = todo & df["dd_path"].str.endswith("/poloidal")
df.loc[rows, "modifier"] = "poloidal"
rows = todo & df["dd_path"].str.endswith("/toroidal")
df.loc[rows, "modifier"] = "toroidal"

rows = todo & df["dd_path"].str.endswith("/diamagnetic")
df.loc[rows, "modifier"] = "diamagnetic"
rows = todo & df["dd_path"].str.endswith("/parallel")
df.loc[rows, "modifier"] = "parallel"

todo = df["standard_name"].isnull()
todo.sum()

## Component limits

In [ ]:
rows = todo & df["dd_path"].str.contains("energy_limit")
df.loc[rows, "standard_name"] = "dissipated_energy"

rows = todo & df["dd_path"].str.contains("current_limit")
df.loc[rows, "standard_name"] = "electric_current"

rows = todo & df["dd_path"].str.contains("voltage_limit")
df.loc[rows, "standard_name"] = "voltage"

rows = todo & df["dd_path"].str.contains("limit_min")
df.loc[rows, "modifier"] = "limit_min"

rows = todo & df["dd_path"].str.contains("limit_max")
df.loc[rows, "modifier"] = "limit_max"

rows = todo & (df["ids_name"] == "pf_active") & df["dd_path"].str.contains("b_field_max")
df.loc[rows, "standard_name"] = "magnetic_field"
df.loc[rows, "modifier"] = "limit_max"

# force_limits structure in pf_active seems under-defined, ignore...
rows = todo & df["dd_path"].str.contains("force_limits/")
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = "force_limits structure seems underdefined, no standard names applied"

display(df.loc[rows])

todo = df["standard_name"].isnull()
todo.sum()


## Voltages and currents on components (actuators / diagnostics / wall)

In [ ]:
rows = todo & df["dd_path"].str.endswith("voltage/data")
df.loc[rows, "standard_name"] = "voltage"

rows = todo & df["dd_path"].str.endswith(("current/data", "loop/current"))
df.loc[rows, "standard_name"] = "electric_current"

rows = todo & (df["dd_path"] == "global_quantities/current_phi")
df.loc[rows, "standard_name"] = "electric_current"
df.loc[rows, "modifier"] = "toroidal in_wall"

rows = todo & (df["dd_path"] == "description_ggd/ggd/j_total")
df.loc[rows, "standard_name"] = "electric_current_density"

todo = df["standard_name"].isnull()
todo.sum()


## Plasma current (density)

In [ ]:
rows = todo & df["dd_path"].str.endswith(("/ip", "ip/data"))
df.loc[rows, "standard_name"] = "plasma_current"

rows = todo & df["dd_path"].str.endswith("j_phi")
df.loc[rows, "standard_name"] = "plasma_current_density"
df.loc[rows, "modifier"] = "toroidal"

rows = todo & df["dd_path"].str.endswith(("j_parallel", "profiles_1d/j_total"))
df.loc[rows, "standard_name"] = "-"
df.loc[rows, "notes"] = """Definitions of j_parallel are slightly confusing:
- in equilibrium/profiles_1d it is an averaged approximation of (j.B)/B0
- in equilibrium/profiles_2d it is (j.B)/B0
- in equilibrium/ggd it is (j.B)/|B|, the actual current density parallel to the
  magnetic field
- in core_profiles/profiles_1d it is called j_total, but defined again as (jtot.B)/B0
"""

display(df.loc[rows])

todo = df["standard_name"].isnull()
todo.sum()

## Pressures

In [ ]:
def ion_neutral_and_states(name):
    """"Gnerate expressions for ion/[state/]name and neutral/[state/]name"""
    return (
        f"ion/{name}", f"ion/state/{name}", f"neutral/{name}", f"neutral/state/{name}"
    )

# Pressure due to all particles
rows = todo & df["dd_path"].str.endswith("profiles_1d/pressure")
df.loc[rows, "standard_name"] = "pressure"

rows = todo & df["dd_path"].str.endswith("d/pressure_thermal")
df.loc[rows, "standard_name"] = "thermal_pressure"

rows = todo & df["dd_path"].str.endswith("d/pressure_parallel")
df.loc[rows, "standard_name"] = "pressure"
df.loc[rows, "modifier"] = "parallel"

rows = todo & df["dd_path"].str.endswith("d/pressure_perpendicular")
df.loc[rows, "standard_name"] = "pressure"
df.loc[rows, "modifier"] = "perpendicular"

# To discuss: technically these are partial pressures, since they describe a pressure
# due to a subset of particles. Howver, this is also clear from the dimensions and we
# can reduce the number of standard names by just choosing "pressure" as standard name
rows = todo & df["dd_path"].str.endswith("d/pressure_ion_total")
df.loc[rows, "standard_name"] = "pressure"
df.loc[rows, "cell_methods"] = "ion: sum"

rows = todo & df["dd_path"].str.endswith("electrons/pressure")
df.loc[rows, "standard_name"] = "pressure"
df.loc[rows, "modifier"] = "of_electrons"

rows = todo & df["dd_path"].str.endswith(ion_neutral_and_states("pressure"))
df.loc[rows, "standard_name"] = "pressure"
# No modifier, dimension clarify which ion (state) this applies to

rows = todo & df["dd_path"].str.endswith("electrons/pressure_thermal")
df.loc[rows, "standard_name"] = "pressure"
df.loc[rows, "modifier"] = "of_electrons thermal"

rows = todo & df["dd_path"].str.endswith("electrons/pressure_fast")
df.loc[rows, "standard_name"] = "pressure"
df.loc[rows, "modifier"] = "of_electrons fast"

rows = todo & df["dd_path"].str.endswith("electrons/pressure_fast_parallel")
df.loc[rows, "standard_name"] = "pressure"
df.loc[rows, "modifier"] = "parallel of_electrons fast"

rows = todo & df["dd_path"].str.endswith("electrons/pressure_fast_perpendicular")
df.loc[rows, "standard_name"] = "pressure"
df.loc[rows, "modifier"] = "perpendicular of_electrons fast"

rows = todo & df["dd_path"].str.endswith(ion_neutral_and_states("pressure_thermal"))
df.loc[rows, "standard_name"] = "pressure"
df.loc[rows, "modifier"] = "thermal"

rows = todo & df["dd_path"].str.endswith(ion_neutral_and_states("pressure_fast"))
df.loc[rows, "standard_name"] = "pressure"
df.loc[rows, "modifier"] = "of_electrons fast"

rows = todo & df["dd_path"].str.endswith(ion_neutral_and_states("pressure_fast_parallel"))
df.loc[rows, "standard_name"] = "pressure"
df.loc[rows, "modifier"] = "parallel fast"

rows = todo & df["dd_path"].str.endswith(ion_neutral_and_states("pressure_fast_perpendicular"))
df.loc[rows, "standard_name"] = "pressure"
df.loc[rows, "modifier"] = "perpendicular fast"

todo = df["standard_name"].isnull()
todo.sum()

## Wall fluxes and recycling coefficients

In [ ]:
# To discuss: should these standard names clarify that they apply to the fluxes incident
# on the wall / emitted by the wall?

# Recycling
rows = todo & df["dd_path"].str.endswith(("/coefficient", "/recycling_particles_coefficient"))
df.loc[rows, "standard_name"] = "particle_recycling_coefficient"  # ?

# Particle fluxes
rows = todo & df["dd_path"].str.contains("particle_fluxes/.*/incident")
df.loc[rows, "standard_name"] = "incident_particle_flux"
rows = todo & df["dd_path"].str.contains("particle_fluxes/.*/emitted")
df.loc[rows, "standard_name"] = "emitted_particle_flux"
rows = todo & df["dd_path"].str.contains("particle_fluxes/electrons/")
df.loc[rows, "modifier"] = "of_electrons"

# Energy fluxes
rows = todo & df["dd_path"].str.endswith("energy_fluxes/radiation/incident")
df.loc[rows, "standard_name"] = "incident_radiation_power_flux"
rows = todo & df["dd_path"].str.endswith("energy_fluxes/radiation/emitted")
df.loc[rows, "standard_name"] = "emitted_radiation_power_flux"

rows = todo & df["dd_path"].str.endswith("energy_fluxes/current/incident")
df.loc[rows, "standard_name"] = "incident_current_power_flux"
rows = todo & df["dd_path"].str.endswith("energy_fluxes/current/emitted")
df.loc[rows, "standard_name"] = "emitted_current_power_flux"

rows = todo & df["dd_path"].str.contains("energy_fluxes/recombination/.*/incident")
df.loc[rows, "standard_name"] = "incident_recombination_power_flux"
rows = todo & df["dd_path"].str.contains("energy_fluxes/recombination/.*/emitted")
df.loc[rows, "standard_name"] = "emitted_recombination_power_flux"

rows = todo & df["dd_path"].str.contains("energy_fluxes/kinetic/.*/incident")
df.loc[rows, "standard_name"] = "incident_kinetic_power_flux"
rows = todo & df["dd_path"].str.contains("energy_fluxes/kinetic/.*/emitted")
df.loc[rows, "standard_name"] = "emitted_kinetic_power_flux"
rows = todo & df["dd_path"].str.contains("energy_fluxes/kinetic/electrons/")
df.loc[rows, "modifier"] = "of_electrons"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# pumping / puff
rows = todo & df["dd_path"].str.endswith("/pumping_speed")
df.loc[rows, "standard_name"] = "pumped_particle_flux"

rows = todo & df["dd_path"].str.endswith("/gas_puff")
df.loc[rows, "standard_name"] = "gas_puff_rate"

# particle fluxes
rows = todo & df["dd_path"].str.endswith("/particle_flux_from_wall")
df.loc[rows, "standard_name"] = "particle_flux_from_wall"
rows = todo & df["dd_path"].str.endswith("/particle_flux_from_plasma")
df.loc[rows, "standard_name"] = "particle_flux_from_plasma"

# Add modifier for electrons
rows = todo & df["dd_path"].str.contains(
    "electrons/(?:pumping_speed|gas_puff|particle_flux_)")
df.loc[rows, "modifier"] = "of_electrons"

todo = df["standard_name"].isnull()
todo.sum()

In [ ]:
# Power fluxes incident to the wall
rows = todo & df["dd_path"].str.endswith(
    ("power_incident", "power_conducted", "power_convected", "power_currents",
     "power_neutrals", "power_radiated", "power_recombination_neutrals",
     "power_recombination_plasma"))
df.loc[rows, "standard_name"] = "net_power_incident_on_wall"

# modifiers for components
df.loc[todo & df["dd_path"].str.endswith("power_conducted"), "modifier"] =              "due_to_conduction"
df.loc[todo & df["dd_path"].str.endswith("power_convected"), "modifier"] =              "due_to_convection"
df.loc[todo & df["dd_path"].str.endswith("power_currents"), "modifier"] =               "due_to_currents"
df.loc[todo & df["dd_path"].str.endswith("power_neutrals"), "modifier"] =               "due_to_neutrals"
df.loc[todo & df["dd_path"].str.endswith("power_radiated"), "modifier"] =               "due_to_radiation"
df.loc[todo & df["dd_path"].str.endswith("power_recombination_neutrals"), "modifier"] = "due_to_neutral_recombination"
df.loc[todo & df["dd_path"].str.endswith("power_recombination_plasma"), "modifier"] =   "due_to_plasma_recombination"


## Atom/ion compositions

In [ ]:
rows = todo & df["dd_path"].str.endswith("element/a")
df.loc[rows, "standard_name"] = "atomic_mass"

rows = todo & df["dd_path"].str.endswith("element/atoms_n")
df.loc[rows, "standard_name"] = "number_of_atoms_per_molecule"  # ?

todo = df["standard_name"].isnull()
todo.sum()


## Rotation frequencies

In [ ]:
rows = todo & df["dd_path"].str.endswith("rotation_frequency_tor")
df.loc[rows, "standard_name"] = "toroidal_rotation_frequency"

todo = df["standard_name"].isnull()
todo.sum()

## Equilibrium flux surface geometries

In [ ]:
rows = todo & df["dd_path"].str.endswith("minor_radius")
df.loc[rows, "standard_name"] = "plasma_shape_minor_radius"

rows = todo & df["dd_path"].str.endswith("elongation")
df.loc[rows, "standard_name"] = "plasma_shape_elongation"

rows = todo & df["dd_path"].str.endswith("triangularity")
df.loc[rows, "standard_name"] = "plasma_shape_triangularity"

rows = todo & df["dd_path"].str.endswith("triangularity_upper")
df.loc[rows, "standard_name"] = "plasma_shape_upper_triangularity"

rows = todo & df["dd_path"].str.endswith("triangularity_lower")
df.loc[rows, "standard_name"] = "plasma_shape_lower_triangularity"

# Squareness
for sqtype in ["upper_inner", "upper_outer", "lower_inner", "lower_outer"]:
    rows = todo & df["dd_path"].str.endswith(f"squareness_{sqtype}")
    df.loc[rows, "standard_name"] = f"plasma_shape_{sqtype}_squareness"

# Add modifier for boundary/*
rows = todo & df["standard_name"].str.startswith("plasma_shape") & df["dd_path"].str.contains("/boundary/")
df.loc[rows, "modifier"] = "at_boundary"

todo = df["standard_name"].isnull()
todo.sum()

## Equilibrium quantities

In [ ]:
# Safety factor
rows = todo & df["dd_path"].str.contains("/q")
df.loc[rows, "standard_name"] = "safety_factor"

df.loc[rows & df["dd_path"].str.endswith("q_axis"), "modifier"] = "at_magnetic_axis"
df.loc[rows & df["dd_path"].str.endswith("q_95"), "modifier"] = "at_95%_poloidal_flux_surface"
df.loc[rows & df["dd_path"].str.endswith("q_min/value"), "cell_methods"] = "safety_factor: minimum"  # ?

todo = df["standard_name"].isnull()
todo.sum()

# beta coefficients
rows = todo & df["dd_path"].str.endswith("/beta_pol")
df.loc[rows, "standard_name"] = "poloidal_beta_coefficient"

rows = todo & df["dd_path"].str.endswith("/beta_tor")
df.loc[rows, "standard_name"] = "toroidal_beta_coefficient"

rows = todo & df["dd_path"].str.endswith("/beta_tor_norm")
df.loc[rows, "standard_name"] = "normalized_toroidal_beta_coefficient"

# internal inductance
rows = todo & df["dd_path"].str.endswith("/li_3")
df.loc[rows, "standard_name"] = "internal_inductance"

# magnetic shear
rows = todo & df["dd_path"].str.endswith("/magnetic_shear")
df.loc[rows, "standard_name"] = "magnetic_shear"

todo = df["standard_name"].isnull()
todo.sum()

## Remaining core_profiles quantities

In [ ]:
#   DD_path:                                    (standard_name,                         modifier,   cell_method)
cp_standard_names = {
    "global_quantities/current_bootstrap":      ("bootstrap_current",                   "toroidal", None),
    "global_quantities/current_non_inductive":  ("non_inductive_current",               "toroidal", None),
    "global_quantities/ejima":                  ("ejima_coefficient",                   None,       None),
    "global_quantities/resistive_psi_losses":   ("resistive_poloidal_flux_losses",      None,       None),
    "global_quantities/t_e_peaking":            ("electron_temperature_peaking_factor", None,       None),
    "global_quantities/t_i_average_peaking":    ("ion_temperature_peaking_factor",      None,       None),
    "global_quantities/v_loop":                 ("lcfs_loop_voltage",                   None,       None),
    "global_quantities/z_eff_resistive":        ("effective_charge_estimate_from_flux_consumption", None, None),
    "profiles_1d/conductivity_parallel":        ("conductivity",                        "parallel", None),
    "profiles_1d/electrons/collisionality_norm": ("ratio_of_collisionality_to_bounce_frequency", None, None),

    # To discuss: is this indeed the plasma current inside the flux surface??
    "profiles_1d/current_parallel_inside":      ("plasma_current",                      "parallel", None),

    # The following ones are flux-surface-averaged, clarify in cell_method:
    "profiles_1d/j_bootstrap":                  ("bootstrap_current_density",           "parallel", "area: mean"),
    "profiles_1d/j_non_inductive":              ("non_inductive_current_density",       "parallel", "area: mean"),
    "profiles_1d/j_ohmic":                      ("ohmic_current_density",               "parallel", "area: mean"),
    "profiles_1d/phi_potential":                ("electrostatic_potential",             None,       "area: mean"),

    "profiles_1d/momentum_phi":                 ("plasma_momentum",                     "toroidal", None),
    "profiles_1d/rotation_frequency_tor_sonic": ("sonic_rotation_frequency",            "toroidal", None),
    "profiles_2d/momentum_phi":                 ("plasma_momentum",                     "toroidal", None),
}

for path, values in cp_standard_names.items():
    rows = todo & (df["ids_name"] == "core_profiles") & (df["dd_path"] == path)
    df.loc[rows, ("standard_name", "modifier", "cell_methods")] = values

todo = df["standard_name"].isnull()
todo.sum()

## Remaining equilibrium quantities

In [ ]:
#   DD_path:                                    (standard_name,                         modifier,   cell_method)
eq_standard_names = {
    "time_slice/boundary/closest_wall_point/distance": 
                                                ("distance_to_wall",                    None,       "distance_to_wall: minimum"),
    # To discuss, perhaps gap descriptions require some more thought
    "time_slice/boundary/gap/angle":            ("poloidal_angle",                      None,       None),
    "time_slice/boundary/gap/value":            ("gap_distance",                        None,       None),
    "time_slice/global_quantities/current_centre/velocity_z":
                                                ("velocity",                            "vertical of_current_centre", None),
    "time_slice/global_quantities/plasma_inductance":
                                                ("plasma_inductance",                   None,       None),
    "time_slice/profiles_1d/darea_dpsi":        ("derivative_of_cross_section_wrt_poloidal_flux", None, None),
    "time_slice/profiles_1d/darea_drho_tor":    ("derivative_of_cross_section_wrt_toroidal_flux_coordinate", None, None),
    "time_slice/profiles_1d/dpressure_dpsi":    ("derivative_of_pressure_wrt_poloidal_flux", None,  None),
    "time_slice/profiles_1d/dpsi_drho_tor":     ("derivative_of_poloidal_flux_wrt_toroidal_flux_coordinate", None, None),
    "time_slice/profiles_1d/dvolume_dpsi":      ("derivative_of_volume_wrt_poloidal_flux", None,    None),
    "time_slice/profiles_1d/dvolume_drho_tor":  ("derivative_of_volume_wrt_toroidal_flux_coordinate", None, None),
    "time_slice/profiles_1d/f":                 ("diamagnetic_function",                None,       None),
    # to discuss: this is much easier, but following CF standard naming scheme we should
    # use: product_of_diamagnetic_function_and_derivative_of_diamagnetic_function_wrt_poloidal_flux
    "time_slice/profiles_1d/f_df_dpsi":         ("f_df_dpsi",                           None,       None),

    # These are all flux-surface averaged:
    # To discuss: following CF conventions, these standard names become ridiculous...
    "time_slice/profiles_1d/gm1":               ("ratio_of_1_to_square_of_radial_distance", None,   "area: mean"),
    "time_slice/profiles_1d/gm2":               ("square_of_ratio_of_magnitude_of_gradient_of_poloidal_flux_coordinate_to_radial_distance", None, "area: mean"),
    "time_slice/profiles_1d/gm3":               ("square_of_magnitude_of_dgradient_of_poloidal_flux_coordinate", None, "area: mean"),
    "time_slice/profiles_1d/gm4":               ("ratio_of_1_to_square_of_magnitude_of_magnetic_field", None, "area: mean"),
    "time_slice/profiles_1d/gm5":               ("square_of_magnitude_of_magnetic_field", None, "area: mean"),
    "time_slice/profiles_1d/gm6":               ("square_of_ratio_of_magnitude_of_gradient_of_poloidal_flux_coordinate_to_magnitude_of_magnetic_field", None, "area: mean"),
    "time_slice/profiles_1d/gm7":               ("magnitude_of_gradient_of_poloidal_flux_coordinate", None, "area: mean"),
    "time_slice/profiles_1d/gm8":               ("radial_distance",                     None,       "area: mean"),
    "time_slice/profiles_1d/gm9":               ("ratio_of_1_to_radial_distance",       None,       "area: mean"),
    "time_slice/profiles_1d/mass_density":      ("mass_density",                        None,       "area: mean"),
    # Flux surface minimum/maximum
    "time_slice/profiles_1d/r_inboard":         ("radial_distance",                     None,       "area: minimum"),
    "time_slice/profiles_1d/r_outboard":        ("radial_distance",                     None,       "area: maximum"),

    "time_slice/profiles_1d/rho_volume_norm":   ("ratio_of_volume_to_volume_at_boundary", None,     None),
    "time_slice/profiles_1d/trapped_fraction":  ("trapped_particle_fraction",           None,       None),
}

for path, values in eq_standard_names.items():
    rows = todo & (df["ids_name"] == "equilibrium") & (df["dd_path"] == path)
    df.loc[rows, ("standard_name", "modifier", "cell_methods")] = values

todo = df["standard_name"].isnull()
todo.sum()

## Remaining magnetics quantities

In [ ]:
# There is only 1 left:

rows = todo & (df["ids_name"] == "magnetics") & (df["dd_path"] == "flux_loop/gm9")
# interesting that this gm9 is different from the one in equilibrium :/
df.loc[rows, "standard_name"] = "integral_of_ratio_of_1_to_radial_distance_wrt_area"

todo = df["standard_name"].isnull()
todo.sum()

## Remaining pf_active quantities

In [ ]:
#   DD_path:                                    (standard_name,                         modifier,   cell_method)
pfa_standard_names = {
    "coil/force_radial/data":                   ("force",                               "radial",   None),
    "coil/force_radial_crushing/data":          ("crushing_force",                      "radial",   None),
    "coil/force_vertical/data":                 ("force",                               "vertical", None),
    "coil/force_vertical_crushing/data":        ("crushing_force",                      "vertical", None),
}

for path, values in pfa_standard_names.items():
    rows = todo & (df["ids_name"] == "pf_active") & (df["dd_path"] == path)
    df.loc[rows, ("standard_name", "modifier", "cell_methods")] = values

todo = df["standard_name"].isnull()
todo.sum()

## Remaining tf quantities

In [ ]:
#   DD_path:                                    (standard_name,                         modifier,   cell_method)
tf_standard_names = {
    # Why do we have both a delta-description and an absolute description? :/
    "b_field_phi_vacuum_r/data":                ("product_of_vacuum_magnetic_field_and_radial_distance", "toroidal", None),
    "delta_b_field_phi_vacuum_r/data":          ("change_over_time_of_product_of_vacuum_magnetic_field_and_radial_distance", "toroidal", None),
}

for path, values in tf_standard_names.items():
    rows = todo & (df["ids_name"] == "tf") & (df["dd_path"] == path)
    df.loc[rows, ("standard_name", "modifier", "cell_methods")] = values

todo = df["standard_name"].isnull()
todo.sum()

## Remaining wall quantities

In [ ]:
#   DD_path:                                    (standard_name,                         modifier,   cell_method)
wall_standard_names = {
    "description_ggd/ggd/phi_potential":        ("electrostatic_potential",             None,       None),
    "description_ggd/ggd/power_density":        ("net_power_density_incident_on_wall",  None,       None),
    "description_ggd/ggd/v_biasing":            ("biasing_potential",                   None,       None),
    "first_wall_power_flux_peak/data":          ("net_power_density_incident_on_wall",  None,       "area: maximum"),
    # TODO:
    # "global_quantities/electrons/power_inner_target":
    # "global_quantities/electrons/power_outer_target":
    # "global_quantities/power_inner_target_ion_total":
    # "global_quantities/neutral/incident_species/energies":
    # "global_quantities/neutral/incident_species/sputtering_chemical_coefficient":
    # "global_quantities/neutral/incident_species/sputtering_physical_coefficient":
    # "global_quantities/neutral/recycling_energy_coefficient":
    # "global_quantities/neutral/wall_inventory":
    "global_quantities/power_black_body":       ("net_power_radiated_by_wall",          None,       None),
    "global_quantities/power_to_cooling":       ("power_to_wall_cooling_system",        None,       None),
}

for path, values in wall_standard_names.items():
    rows = todo & (df["ids_name"] == "wall") & (df["dd_path"] == path)
    df.loc[rows, ("standard_name", "modifier", "cell_methods")] = values

todo = df["standard_name"].isnull()
todo.sum()

# Exploration

In [ ]:
print("\n".join(df.loc[todo, "dd_path"]))
display(df[todo])

# Analysis

## Defined standard names

In [ ]:
# standard names
print("Qtys with standard name:", df["standard_name"].str.contains("-|<").value_counts()[False])
with pandas.option_context('display.max_rows', None):
    cts = df["standard_name"].value_counts()
    print(f"Number of unique standard names: {len(cts)}")
    display(cts.to_frame().sort_values(["count", "standard_name"], ascending=[False, True]))

## Modifiers used

In [ ]:
cts = (
    # Take all modifiers that are not None
    df["modifier"].dropna()
    # Some modifiers are a combination of things (separated by whitespace)
    # separate them:
    .str.split().explode()
    # get unique values and their counts
    .value_counts().to_frame().sort_values("modifier")
)
print("Number of modifiers: ", len(cts))
print("Applied to #quantities:", df["modifier"].count())
print("Applied to #quantities (-error):", df["modifier"].count() - (df["modifier"].dropna() == "standard_error").value_counts()[True])
cts

In [ ]:
df.loc[df["modifier"].str.contains("of_ion", na=False)]

## Cell methods

In [ ]:
cell_methods = (
    # Take all cell_methods that are not None
    df["cell_methods"].dropna()
    # Some have multiple cell methods, split them based on whitespace not following a `:`
    .str.split(r"(?<!:)\s+").explode()
    # Count unique values and sort
    .value_counts().to_frame().sort_values("cell_methods")
)
cell_methods

In [ ]:
df.loc[~df["cell_methods"].isnull()].sort_values("cell_methods")

## Floating point, non-error quantities without standard names

In [ ]:
filtered = df.query("(standard_name == '-') & (data_type == 'FLT')")
# Remove columns that we're not interested in now
filtered = filtered.drop(["structure_reference", "standard_name", "modifier", "cell_methods"], axis=1)
# set a multi-index
filtered = filtered.reset_index().set_index(["notes", "ids_name", "dd_path"]).sort_index().drop("index", axis=1)

print("Number of floating point, non-error quantities without standard names:", len(filtered))

with pandas.option_context('display.max_rows', None):
    display(filtered)

# Export

In [ ]:
# df.to_html("standard_names.html")
# df.to_csv("standard_names.csv")

# Queries

In [ ]:
df.loc[df["standard_name"].str.contains("divergence_of", na=False)]